# Phenotype Generation: Number of Therapies, Substances and Prescriptions

## Phenotype Processing Methodology

The following phenotypes are **count data** (number of therapies, prescriptions, substances). To facilitate robust statistical analysis and prevent single extreme values from dominating results, a uniform methodology is applied for outlier capping and transformation across all listed phenotypes.

| Category | Outlier Capping Method | Transformation Method |
| :--- | :--- | :--- |
| **All Count Phenotypes** | The maximum value for each phenotype is capped at $\mathbf{\mu + 8\sigma}$ (mean plus eight standard deviations), calculated independently *per* phenotype. | **Box-Cox Transformation** |

## Detailed Phenotype Definitions

| **Phenotype Name** | **Definition** | **Transformed Phenotype Name** |
| :--- | :--- | :--- |
| **`<drug_name>__num_of_prescriptions`** | Number of prescriptions for a specific drug for an individual. | **`<drug_name>__num_of_prescriptions__box_cox`** |
| **`<drug_name>__num_of_therapies`** | Number of unique therapies for a specific drug for an individual. | **`<drug_name>__num_of_therapies__box_cox`** |
| **`<bnf_section_short_name>__num_of_substances`** | Number of unique substances prescribed within a given BNF section for an individual. | **`<bnf_section_short_name>__num_of_substances__box_cox`** |
| **`<bnf_section_short_name>__num_of_prescriptions`** | Number of prescriptions within a given BNF section for an individual. | **`<bnf_section_short_name>__num_of_prescriptions__box_cox`** |
| **`<bnf_section_short_name>__num_of_therapies`** | Number of unique therapies within a given BNF section for an individual. | **`<bnf_section_short_name>__num_of_therapies__box_cox`** |

short names mappings are available in: `../../../data/input/bnf_sections.csv`




In [ ]:
import pyspark
import dxpy
import hail as hl
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import sys
import json
from scipy.stats import boxcox

In [ ]:
sys.path.append('../')
from functions.bnf_dictionaries_utils import prepare_substance_to_bnf_section_code_dict, prepare_bnf_section_code_to_short_name_dict
from functions.phenotype_filtration_and_normalization import cap_outliers_mu_plus_n_sigma, pivot_multiple_phenotypes, phenotype_box_cox_transformation

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Configuration and Hail tables loading

In [ ]:
input_database = 'prescriptions_db'
input_prescriptions_tb = 'cleaned_prescriptions_splited_to_therapies_v6.2.0.ht'

output_database = 'prescriptions_db'
sections_output_tb = 'count_phenotypes_section_v6_2_0.ht'
substances_output_tb = 'count_phenotypes_substance_v6_2_0.ht'

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
ht = hl.read_table(f'dnax://{input_db_id}/{input_prescriptions_tb}')
ht.describe()

In [ ]:
ht = ht.explode('prescriptions')
ht = ht.persist()

## Substances phenotypes

In [ ]:
ht_substances_phenotypes = ht.group_by(
    ht.eid,
    ht.substance
).aggregate(
    num_of_prescriptions = hl.agg.count(),
    num_of_therapies = hl.len(hl.agg.collect_as_set(ht.tid))
)
ht_substances_phenotypes = ht_substances_phenotypes.persist()

In [ ]:
ht_substances_phenotypes = cap_outliers_mu_plus_n_sigma(
    ht=ht_substances_phenotypes, 
    grouping_col_name='substance', 
    phenotype_col_name='num_of_prescriptions',
    capped_col_name='capped_num_of_prescriptions'
)

ht_substances_phenotypes = cap_outliers_mu_plus_n_sigma(
    ht=ht_substances_phenotypes, 
    grouping_col_name='substance', 
    phenotype_col_name='num_of_therapies',
    capped_col_name='capped_num_of_therapies'
)

ht_substances_phenotypes = ht_substances_phenotypes.persist()

In [ ]:
ht_substances_phenotypes = phenotype_box_cox_transformation(
    ht=ht_substances_phenotypes, 
    grouping_col_name='substance',
    phenotype_col_name='capped_num_of_prescriptions',
    transformed_col_name='box_cox_num_of_prescriptions'
)

ht_substances_phenotypes = phenotype_box_cox_transformation(
    ht=ht_substances_phenotypes, 
    grouping_col_name='substance',
    phenotype_col_name='capped_num_of_therapies',
    transformed_col_name='box_cox_num_of_therapies'
)

In [ ]:
final_ht = pivot_multiple_phenotypes(
    ht=ht_substances_phenotypes, 
    key_col_name='eid', 
    pivot_col_name='substance', 
    phenotype_mappings=[
        ('capped_num_of_prescriptions', '_num_of_prescriptions'),
        ('capped_num_of_therapies', '_num_of_therapies'),
        ('box_cox_num_of_prescriptions', '_num_of_prescriptions__box_cox'),
        ('box_cox_num_of_therapies', '_num_of_therapies__box_cox')
    ]
)

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_database} LOCATION 'dnax://'")
output_db_id = dxpy.find_one_data_object(name=output_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
substances_output_tb_url = f'dnax://{output_db_id}/{substances_output_tb}'

%time final_ht.write(substances_output_tb_url, overwrite=True)

## Sections phenotypes

### Substances to section code dictionary preparation

In [ ]:
substances_section_code_dict = prepare_substance_to_bnf_section_code_dict('../../../data/input/drug_list.csv')
substances_section_code_dict_hl = hl.literal(substances_section_code_dict)

In [ ]:
code_to_short_name_dict = prepare_bnf_section_code_to_short_name_dict('../../../data/input/bnf_sections.csv')

In [ ]:
code_to_short_name_dict_hl = hl.literal(code_to_short_name_dict)

In [ ]:
grouped_ht = ht.group_by(ht.eid, ht.substance, ht.tid).aggregate(
    collected_data = hl.agg.collect(
        hl.struct(
            date_info=ht.date_struct,
            interval=ht.interval
        )
    )
)
grouped_ht = grouped_ht.annotate(
    sorted_collected_data = hl.sorted(
        grouped_ht.collected_data,
        key=lambda s: (s.date_info.year, s.date_info.month, s.date_info.day)
    )
).drop('collected_data')
grouped_ht = grouped_ht.persist()

In [ ]:
ht = grouped_ht.annotate(
    first_date_struct = grouped_ht.sorted_collected_data[0].date_info,
    last_date_struct = grouped_ht.sorted_collected_data[-1].date_info
).drop('sorted_collected_data')
ht = ht.persist()

ht = ht.annotate(
    bnf_section_codes = substances_section_code_dict_hl.get(ht.substance, hl.empty_array('str'))
)
ht = ht.persist()

ht = ht.explode(ht.bnf_section_codes)
ht_sections_phenotypes = ht.group_by(
    ht.eid,
    ht.bnf_section_codes
).aggregate(
    num_of_prescriptions = hl.agg.count(),
    num_of_substances = hl.len(hl.agg.collect_as_set(ht.substance)),
    num_of_therapies = hl.len(hl.agg.collect_as_set(ht.tid)),
)
ht_sections_phenotypes = ht_sections_phenotypes.key_by().persist()

In [ ]:
ht_exploded = ht_sections_phenotypes.annotate(
    bnf_section_short_name = code_to_short_name_dict_hl.get(ht_sections_phenotypes.bnf_section_codes, hl.empty_array('str'))
).drop('bnf_section_codes')
ht_exploded = ht_exploded.explode(ht_exploded.bnf_section_short_name)
ht_exploded = ht_exploded.persist()

In [ ]:
ht_sections_phenotypes = cap_outliers_mu_plus_n_sigma(
    ht=ht_exploded, 
    grouping_col_name='bnf_section_short_name', 
    phenotype_col_name='num_of_prescriptions',
    capped_col_name='capped_num_of_prescriptions'
)

ht_sections_phenotypes = cap_outliers_mu_plus_n_sigma(
    ht=ht_sections_phenotypes, 
    grouping_col_name='bnf_section_short_name', 
    phenotype_col_name='num_of_substances',
    capped_col_name='capped_num_of_substances'
)

ht_sections_phenotypes = cap_outliers_mu_plus_n_sigma(
    ht=ht_sections_phenotypes, 
    grouping_col_name='bnf_section_short_name', 
    phenotype_col_name='num_of_therapies',
    capped_col_name='capped_num_of_therapies'
)

ht_sections_phenotypes = ht_sections_phenotypes.persist()

In [ ]:
ht_sections_phenotypes = phenotype_box_cox_transformation(
    ht=ht_sections_phenotypes, 
    grouping_col_name='bnf_section_short_name',
    phenotype_col_name='capped_num_of_prescriptions',
    transformed_col_name='box_cox_num_of_prescriptions'
)

ht_sections_phenotypes = phenotype_box_cox_transformation(
    ht=ht_sections_phenotypes, 
    grouping_col_name='bnf_section_short_name',
    phenotype_col_name='capped_num_of_therapies',
    transformed_col_name='box_cox_num_of_therapies'
)

ht_sections_phenotypes = phenotype_box_cox_transformation(
    ht=ht_sections_phenotypes, 
    grouping_col_name='bnf_section_short_name',
    phenotype_col_name='capped_num_of_substances',
    transformed_col_name='box_cox_num_of_substances'
)

In [ ]:
final_ht = pivot_multiple_phenotypes(
    ht=ht_sections_phenotypes, 
    key_col_name='eid', 
    pivot_col_name='bnf_section_short_name', 
    phenotype_mappings=[
        ('capped_num_of_prescriptions', '_num_of_prescriptions'),
        ('capped_num_of_therapies', '_num_of_therapies'),
        ('capped_num_of_substances', '_num_of_substances'),
        ('box_cox_num_of_prescriptions', '_num_of_prescriptions__box_cox'),
        ('box_cox_num_of_therapies', '_num_of_therapies__box_cox'),
        ('box_cox_num_of_substances', '_num_of_substances__box_cox')
    ]
)

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_database} LOCATION 'dnax://'")
output_db_id = dxpy.find_one_data_object(name=output_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
sections_output_tb_url = f'dnax://{output_db_id}/{sections_output_tb}'

%time final_ht.write(sections_output_tb_url, overwrite=True)